![](../assets/placeholder.png)

## Giriş
Bu notebook, Langchain ve OpenAI'nin GPT-4 LLM'i kullanarak yapısız metinden yapılandırılmış bilgileri tanımlayan ve çıkaran bir şema çıkarma uygulaması oluşturma sürecinde sizi yönlendirmek için tasarlanmıştır. Şema çıkarma, verileri daha erişilebilir bir biçimde anlaşılması ve organize edilmesi için özellikle yararlı olabilir.

In [ ]:
from dotenv import load_dotenv
import os

# .env dosyasından ortam değişkenlerini yükleyin
load_dotenv()

In [ ]:
from typing import List, Optional

from langchain_core.pydantic_v1 import BaseModel, Field


class Agent_Feature(BaseModel):
    """AI ajanlarının özelliği hakkında bilgi. AI Ajanlar gerçekten harika bir yetenektir. Çeşitli özelliklerden oluşur, bu bunları tanımlar"""

    # ^ Agent_Feature varlığı için belge dizesi.
    # Bu belge dizesi Agent_Feature şemasının açıklaması olarak LLM'ye gönderilir,
    # ve çıkarma sonuçlarını iyileştirmeye yardımcı olabilir.

    # Not:
    # 1. Her alan `isteğe bağlı` -- bu modelin çıkarmayı reddetmesini sağlar!
    # 2. Her alan bir `açıklama` olur -- bu açıklama LLM tarafından kullanılır.
    # İyi bir açıklamaya sahip olmak çıkarma sonuçlarını iyileştirmeye yardımcı olabilir.
    name: Optional[str] = Field(default=None, description="Aracı özelliğinin adı")
    definition: Optional[str] = Field(default=None, description="AI ajanlarının bu özelliğinin kısa bir tanımı")


class Data(BaseModel):
    """AI Aracı Özellikleri hakkında alınan veriler."""

    # Birden fazla varlığı çıkarabileceğimiz şekilde bir model oluşturur.
    features: List[Agent_Feature]

In [ ]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.pydantic_v1 import BaseModel, Field


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Siz metin metininden ilgili bilgileri çıkaran bir uzman çıkarma algoritmasısınız. "
            "Yalnızca metinden ilgili bilgileri çıkarın. "
            "Çıkarılan bir özniteliğin değerini bilmiyorsanız, "
            "özniteliğin değeri için null döndürün.",
        ),
        ("human", "{text}"),
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o")

In [ ]:
runnable = prompt | model.with_structured_output(schema=Data)
text = "Araç kullanımı AI ajanlarının temel bir yeteneğidir. Ajanı metodları çağıran araçları çağırmasına izin verir, API'ları arayabilir, docsları okuyabilir, işlemler gerçekleştirebilir ve benzeri işlemler yapabilir."
runnable.invoke({"text": text})

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
loader = WebBaseLoader(url)
docs = loader.load()

In [ ]:
runnable.invoke({"text": docs})